## 1. Общий setup

Заполните `REPO_URL`, при необходимости поменяйте `BRANCH`, затем выполните блок. CSV-файлы должны лежать в корне репозитория.

In [ ]:
# 1. Общий setup
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Timur713/HSE_RAG_Hackaton"  # pure URL or markdown link are both accepted
BRANCH = "main"
PROJECT_DIR = Path("/content/legal_hse_4")
RUN_OPTIONAL_DENSE = False  # True installs sentence-transformers and enables dense configs


def normalize_repo_url(value: str) -> str:
    value = str(value).strip()
    markdown_match = re.search(r"\((https?://[^)]+|git@[^)]+)\)", value)
    if markdown_match:
        value = markdown_match.group(1)
    else:
        plain_match = re.search(r"https?://[^\s\]]+|git@[^\s\]]+", value)
        if plain_match:
            value = plain_match.group(0)
    value = value.strip().strip("'").strip('"')
    if value.endswith("/"):
        value = value[:-1]
    return value


def run_cmd(cmd, *, cwd=None, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(map(str, cmd))}")
    return result


REPO_URL = normalize_repo_url(REPO_URL)
if not REPO_URL:
    raise ValueError("Заполните REPO_URL чистым URL GitHub-репозитория.")
print("Using repo:", REPO_URL)

# Colab workspace is disposable. Re-cloning is more reliable than trying to repair a dirty git checkout.
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

clone_result = run_cmd(["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=False)
if clone_result.returncode != 0:
    print(f"Clone with --branch {BRANCH!r} failed; retrying default branch clone.")
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    run_cmd(["git", "clone", REPO_URL, str(PROJECT_DIR)])
    branches = run_cmd(["git", "-C", str(PROJECT_DIR), "branch", "-a"], check=False)
    if f"remotes/origin/{BRANCH}" in branches.stdout:
        run_cmd(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH])
    else:
        print(f"Branch {BRANCH!r} not found; using repository default branch.")

run_cmd([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_DIR)])
if RUN_OPTIONAL_DENSE:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements-optional.txt")])

os.chdir(PROJECT_DIR)
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from legal_hse.config import PathConfig

paths = PathConfig.from_root(PROJECT_DIR)
paths.ensure_dirs()
paths.require_input_files()
print(f"Project ready: {PROJECT_DIR}")


## 2. Все эксперименты

`cv` используйте для перебора конфигураций на train-pool. `holdout` запускайте отдельно только для финальной проверки выбранных экспериментов.

In [ ]:
# 2a. Focused sweep: токенизация + лемматизация для sparse retrieval
import sys
from legal_hse.experiments import default_experiments

# pymorphy3 нужен только для lemmatize=True; без него код fallback'ится в обычные токены.
run_cmd([sys.executable, "-m", "pip", "install", "-q", "pymorphy3>=2.0.6"])

TOKENIZATION_EXPERIMENT_NAMES = [
    # controls from the current strong sparse baseline
    "bm25_doc",
    "bm25_chunk_line_10_5_max",
    "tfidf_char_doc_3_5",
    "rrf_sparse_doc_chunk_char",
    # pure lemmatization checks
    "tfidf_word_lemma_doc",
    "bm25_lemma_doc",
    # legal-aware tokenizer: keeps ст. 333.19, 75-ФЗ, НК/ГК/РФ and removes ФИО/адрес noise
    "tfidf_word_legal_lemma_doc",
    "bm25_legal_lemma_doc",
    "bm25_legal_lemma_chunk_line_10_5_max",
    "rrf_legal_lemma_doc_chunk",
    "rrf_sparse_legal_lemma_char",
    "rrf_sparse_legal_lemma_word_char",
    # phrase variant for stable legal collocations; keep only if validation beats simpler lemmas
    "bm25_legal_phrase_doc",
    "bm25_legal_phrase_chunk_line_10_5_max",
    "rrf_legal_phrase_doc_chunk",
]

available_experiments = {spec.name for spec in default_experiments(include_optional=RUN_OPTIONAL_DENSE)}
missing = sorted(set(TOKENIZATION_EXPERIMENT_NAMES) - available_experiments)
if missing:
    raise ValueError(f"Unknown experiments: {missing}")

MODE = "cv"
EXPERIMENT_NAMES = TOKENIZATION_EXPERIMENT_NAMES
SEED = 42
N_SPLITS = 5
print("Tokenization/lemmatization sweep:")
for name in EXPERIMENT_NAMES:
    print("-", name)


In [ ]:
# 2. Все эксперименты
from legal_hse.experiments import run_suite, select_best_experiment

MODE = globals().get("MODE", "cv")  # "holdout", "cv" or "train"
EXPERIMENT_NAMES = globals().get("EXPERIMENT_NAMES", None)  # None runs default suite
SEED = globals().get("SEED", 42)
N_SPLITS = globals().get("N_SPLITS", 5)

summary = run_suite(
    data_dir=PROJECT_DIR,
    output_dir=PROJECT_DIR / "reports",
    experiment_names=EXPERIMENT_NAMES,
    mode=MODE,
    include_optional=RUN_OPTIONAL_DENSE,
    seed=SEED,
    n_splits=N_SPLITS,
    eval_depth=50,
)

cols = [
    "experiment",
    "status",
    "n_splits",
    "train_recall@5_mean",
    "train_recall@5_std",
    "valid_recall@5_mean",
    "valid_recall@5_std",
    "valid_recall@10_mean",
    "valid_recall@20_mean",
    "valid_recall@50_mean",
    "holdout_recall@5_mean",
    "holdout_recall@5_std",
    "holdout_recall@10_mean",
    "holdout_recall@20_mean",
    "holdout_recall@50_mean",
]
visible_cols = [c for c in cols if c in summary.columns]
sort_col = next(c for c in ["holdout_recall@5_mean", "valid_recall@5_mean", "train_recall@5_mean"] if c in summary.columns)
display(summary[visible_cols].sort_values(["status", sort_col], ascending=[True, False]))
BEST_EXPERIMENT = select_best_experiment(summary)
print("BEST_EXPERIMENT =", BEST_EXPERIMENT)

## 3. Загрузка результатов на GitHub

Введите `GITHUB_USERNAME`, `GIT_EMAIL`, `SSH_PRIVATE_KEY_B64`. Ключ должен иметь write-доступ к репозиторию.

In [ ]:
# 3. Загрузка результатов [метрик] на GitHub
import base64
import getpass
import os
import subprocess
from pathlib import Path

GITHUB_USERNAME = input("GITHUB_USERNAME: ").strip()
GIT_EMAIL = input("GIT_EMAIL: ").strip()
SSH_PRIVATE_KEY_B64 = getpass.getpass("SSH_PRIVATE_KEY_B64: ").strip()
SSH_REMOTE_URL = ""  # optional: git@github.com:<user>/<repo>.git

ssh_dir = Path.home() / ".ssh"
ssh_dir.mkdir(mode=0o700, exist_ok=True)
key_path = ssh_dir / "id_ed25519"
key_path.write_bytes(base64.b64decode(SSH_PRIVATE_KEY_B64))
key_path.chmod(0o600)
subprocess.run(["ssh-keyscan", "github.com"], stdout=(ssh_dir / "known_hosts").open("a"), check=True)

subprocess.run(["git", "config", "user.name", GITHUB_USERNAME], check=True)
subprocess.run(["git", "config", "user.email", GIT_EMAIL], check=True)

if not SSH_REMOTE_URL and REPO_URL.startswith("https://github.com/"):
    repo_path = REPO_URL.split("https://github.com/", 1)[1]
    repo_path = repo_path[:-4] if repo_path.endswith(".git") else repo_path
    SSH_REMOTE_URL = f"git@github.com:{repo_path}.git"
if SSH_REMOTE_URL:
    subprocess.run(["git", "remote", "set-url", "origin", SSH_REMOTE_URL], check=True)

paths_to_add = ["reports/metrics", "reports/summary_latest.csv"]
paths_to_add += [str(path) for path in Path("reports").glob("summary_*.csv")]
subprocess.run(["git", "add", *paths_to_add], check=True)
status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, check=True).stdout.strip()
if status:
    subprocess.run(["git", "commit", "-m", "Add retrieval experiment metrics"], check=True)
    subprocess.run(["git", "push", "origin", f"HEAD:{BRANCH}"], check=True)
    print("Metrics pushed to GitHub.")
else:
    print("No metric changes to push.")

## 4. Формирование submission файла

По умолчанию используется лучший experiment name из блока 2. Его можно заменить вручную.

In [ ]:
# 4. Формирование submission файла
import pandas as pd
from legal_hse.experiments import create_submission

FINAL_EXPERIMENT = globals().get("BEST_EXPERIMENT", "rrf_bm25_doc_chunk")
SUBMISSION_PATH = create_submission(
    data_dir=PROJECT_DIR,
    experiment_name=FINAL_EXPERIMENT,
    output_path=PROJECT_DIR / "submissions" / f"submission_{FINAL_EXPERIMENT}.csv",
    include_optional=RUN_OPTIONAL_DENSE,
    top_k=5,
)
print("Submission:", SUBMISSION_PATH)
display(pd.read_csv(SUBMISSION_PATH).head(15))